# inference

## setup

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import os
import torch
from tqdm import tqdm
# preprocessing
from transformers import (
     pipeline,
)
import spacy
# ml
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    make_scorer,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import cross_val_score, ShuffleSplit, train_test_split, GridSearchCV
from sklearn import svm
from sklearn import metrics
#plotting
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt

In [2]:
## global variables and setup
MODEL = "../../projects/classifier/spatial-entities-classifier/models/modern-gbert-binary-nnse-best/"
CORPUS_PATH = "../corpora-collection/sunset-corpus/src/normalized/"
MAX_CHUNKS = 300
pio.templates.default="gridon"

In [3]:
## spacy
nlp = spacy.load("de_dep_news_trf", 
                 exclude=[
        "tagger",
        "morphologizer",
        "lemmatizer",
        "attribute_ruler",
    ],)

In [4]:
## load trained model
classifier = pipeline(
    task="token-classification",
    model=MODEL,
    tokenizer=MODEL,
    aggregation_strategy="none",
    device=0 if torch.cuda.is_available() else -1,
)

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

In [5]:
# helper functions
def split_into_chunks(text: str, chunk_size: int = 300) -> list[str]:
    words = text.split()

    return [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), chunk_size)
    ]
def join_subwords(tokens: list[str]) -> list[str]:
    result = []

    for token in tokens:
        if token.startswith("##") and result:
            result[-1] += token[2:]
        else:
            result.append(token)

    return result
def create_document_pca(
    z_df: pd.DataFrame,
) -> tuple[pd.DataFrame, PCA]:
    """
    Reduce a document matrix to two PCA dimensions.

    Expected input:
        rows    = words
        columns = documents
        values  = z-scores
    """
    # Documents become rows; words become features.
    document_matrix = z_df.T

    if document_matrix.shape[0] < 2:
        raise ValueError("At least two documents are required.")

    if document_matrix.shape[1] < 2:
        raise ValueError("At least two word features are required.")

    pca = PCA(n_components=2)

    embedding = pca.fit_transform(document_matrix)

    pca_df = pd.DataFrame({
        "document": document_matrix.index,
        "pca_1": embedding[:, 0],
        "pca_2": embedding[:, 1],
    })

    return pca_df, pca

In [6]:
class WordCountMatrix:
    def __init__(self):
        self.vocabulary: list[str] = []
        self._known_words: set[str] = set()
        self.documents: list[Counter] = []
        self.column_names: list[str] = []

    def update(
        self,
        words: list[str],
        column_name: str,
    ) -> None:
        """Add one word list as a named matrix column."""
        if column_name in self.column_names:
            raise ValueError(
                f"Column name already exists: {column_name}"
            )

        self.documents.append(Counter(words))
        self.column_names.append(column_name)

        for word in words:
            if word not in self._known_words:
                self._known_words.add(word)
                self.vocabulary.append(word)

    def update_many(
        self,
        word_lists: list[list[str]],
        column_names: list[str],
    ) -> None:
        """Add several named word lists."""
        if len(word_lists) != len(column_names):
            raise ValueError(
                "word_lists and column_names must have the same length."
            )

        for words, column_name in zip(word_lists, column_names):
            self.update(words, column_name)

    def to_dataframe(self) -> pd.DataFrame:
        """Return words as rows and word lists as columns."""
        data = {
            column_name: [
                document.get(word, 0)
                for word in self.vocabulary
            ]
            for column_name, document in zip(
                self.column_names,
                self.documents,
            )
        }

        return pd.DataFrame(
            data,
            index=self.vocabulary,
            dtype=float,
        )

    @staticmethod
    def _z_transform(
        df: pd.DataFrame,
        axis: int = 0,
    ) -> pd.DataFrame:
        """
        Apply a z-transformation.

        axis=0:
            Standardize each document column across words.

        axis=1:
            Standardize each word across documents.
        """
        if axis == 0:
            means = df.mean(axis=0)
            stds = df.std(axis=0, ddof=0).replace(0, 1)

            return (df - means) / stds

        if axis == 1:
            means = df.mean(axis=1)
            stds = df.std(axis=1, ddof=0).replace(0, 1)

            return df.sub(means, axis=0).div(stds, axis=0)

        raise ValueError("axis must be 0 or 1.")

    def to_z_dataframe(
        self,
        axis: int = 0,
    ) -> pd.DataFrame:
        """Return the complete z-transformed matrix."""
        df = self.to_dataframe()
        return self._z_transform(df, axis=axis)

    def most_frequent(
        self,
        n: int,
        z_transform: bool = False,
        axis: int = 0,
    ) -> pd.DataFrame:
        """
        Return the n most frequent words.

        Frequencies are calculated from the raw count matrix.
        The selected rows can optionally be z-transformed.
        """
        if n < 1:
            raise ValueError("n must be at least 1.")

        df = self.to_dataframe()

        # Select words using their raw total frequency.
        total_counts = df.sum(axis=1)
        selected_words = total_counts.nlargest(n).index

        result = df.loc[selected_words].copy()

        if z_transform:
            result = self._z_transform(
                result,
                axis=axis,
            )

        return result

    def to_numpy(self):
        """Return the complete count matrix as a NumPy array."""
        return self.to_dataframe().to_numpy()
    
    @classmethod
    def from_dataframe(
        cls,
        df: pd.DataFrame,
        ) -> "WordCountMatrix":
        """
        Create a WordCountMatrix from a pandas DataFrame.
    
        Expected format:
            rows    = words
            columns = documents
            values  = word counts
        """
        # Check for duplicate words or document names
        if df.index.has_duplicates:
            raise ValueError(
                "DataFrame index contains duplicate words."
            )
    
        if df.columns.has_duplicates:
            raise ValueError(
                "DataFrame contains duplicate column names."
            )
    
        # Check that all values are numeric
        if not all(
            pd.api.types.is_numeric_dtype(dtype)
            for dtype in df.dtypes
        ):
            raise ValueError(
                "All DataFrame values must be numeric."
            )
    
        # Create empty object
        matrix = cls()
    
        # Restore vocabulary
        matrix.vocabulary = df.index.tolist()
        matrix._known_words = set(matrix.vocabulary)
    
        # Restore document names
        matrix.column_names = df.columns.tolist()
    
        # Restore Counters
        for column_name in matrix.column_names:
            counts = df[column_name]
    
            document = Counter({
                word: count
                for word, count in counts.items()
                if count != 0
            })
    
            matrix.documents.append(document)

        return matrix

## train model

In [7]:
df = pd.read_csv("src/city_country_metadata.tsv", sep="\t")
len(df)

156

In [8]:
# counter.to_dataframe().to_csv("src/results_raw.tsv", sep="\t", index=True)
counter = WordCountMatrix.from_dataframe(pd.read_csv("src/results_raw.tsv", sep="\t", index_col="Unnamed: 0"))

In [9]:
mat = counter.most_frequent(
        n=300,
        z_transform=True,
        axis=0,
    )
    
X_train, X_test, y_train, y_test = train_test_split(mat.T.to_numpy(),
                                                    df.textClass,
                                                    test_size=0.20,
                                                    random_state=42,
                                                    stratify=df.textClass)

In [10]:
model = svm.SVC(
    kernel="rbf",
    C=10,
    gamma=0.01,
)

In [11]:
model.fit(X_train, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",10
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.01
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [12]:
y_pred = model.predict(X_test)

In [13]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       Other       0.80      1.00      0.89         8
       Rural       0.85      0.85      0.85        13
       Urban       1.00      0.82      0.90        11

    accuracy                           0.88        32
   macro avg       0.88      0.89      0.88        32
weighted avg       0.89      0.88      0.88        32



## load sunset dataset

In [14]:
meta = pd.read_csv("../../GitHub/corpora-collection/sunset-corpus/metadata_updated-v2.csv")
meta.columns

Index(['filename', 'ID', 'title', 'gndID', 'author', 'birthYear', 'deathYear',
       'gender', 'nationality', 'pubYear', 'numTokens', 'timeslotPubYear',
       'timeslotBirthYear', 'source', 'n_chunks', 'textClass',
       'mean_prob_prose', 'mean_prob_drama', 'mean_prob_lyric',
       'corpusSubset'],
      dtype='str')

In [16]:
inference_data = meta.loc[~meta.filename.isin(df.filename)]

In [17]:
# header = "filename\tprediction\n"
# with open("src/results_predictions_sunset.tsv", "w") as f:
#     f.write(header)

for file in tqdm(inference_data.filename):
    if file in list(pd.read_csv("src/results_predictions_sunset.tsv", sep ="\t").filename):
        continue
    
    #open text
    with open(CORPUS_PATH + file) as f:
        text = f.read()
    
    # create WordCountMatrix
    #chunk text
    chunks = split_into_chunks(text=text, chunk_size=300)
    # only look at beginning of text
    max_chunk_size = MAX_CHUNKS
    if len(chunks)>max_chunk_size:
        chunks_to_process = chunks[:max_chunk_size]
    else:
        chunks_to_process = chunks
    # create sentences
    docs = nlp.pipe(chunks_to_process, batch_size=32, n_process=1)
    sentences = [
        sent.text
        for doc in docs
        for sent in doc.sents
    ]
    # extract nnse
    predictions = classifier(sentences)
    words_raw = [
        prediction["word"]
        for document_predictions in predictions
        for prediction in document_predictions
    ]
    words = join_subwords(words_raw)
    # update WordCountMatrix
    counter.update(words, column_name=file)

    # create_data
    matrix = counter.most_frequent(
        n=300,
        z_transform=True,
        axis=0,
    )
    prediction = model.predict(matrix[file].T.to_numpy().reshape(1, -1))[0]
    row = f"{file}\t{prediction}\n"

    with open("src/results_predictions_sunset.tsv", "a") as f:
        f.write(row)

  2%|▌                                   | 105/6227 [43:41<42:27:48, 24.97s/it]


KeyboardInterrupt: 